# Output Curves → Dashboard

Procesa curvas de salida desde HDF5, genera CSV con datos crudos para el dashboard y registra en SQLite.

**Flujo:**
1. Seleccionar archivo HDF5
2. Leer todas las curvas output (I_DS vs V_DS para diferentes V_GS)
3. Generar CSV con datos crudos + dirección del sweep
4. Visualizar gráficas
5. Guardar CSV en `data/processed/`
6. Registrar en SQLite (memoria + referencias + análisis)

**Nota:** Este notebook genera datos crudos. El análisis de parámetros físicos (movilidad lineal, resistencia) se puede hacer posteriormente.

In [ ]:
import os, sys
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root(marker='src'):
    cwd = Path(os.getcwd()).resolve()
    for parent in [cwd, *cwd.parents]:
        if (parent / marker).is_dir():
            return parent
    raise FileNotFoundError(f"No se encontro project root con '{marker}'")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from src.readers.keithley_dean import list_measurements, read_curve, read_all_curves
from src.plotting import plot_output
from src import db

db.init_db()
print('Project root:', PROJECT_ROOT)
print('Database initialized at', db.DB_PATH)

In [ ]:
data_folder = PROJECT_ROOT / 'data' / 'raw'

print('Archivos HDF5 disponibles:')
hdf5_files = sorted([f for f in os.listdir(data_folder) if f.endswith('.hdf5')])
for f in hdf5_files:
    print(f'  {f}')

# === MODIFICAR: Seleccionar archivo HDF5 ===
hdf5_name = '2026-09-04_kapton_test_out.hdf5'
hdf5_path = PROJECT_ROOT / 'data' / 'raw' / hdf5_name

print('\n' + '='*80)
print(f'Archivo seleccionado: {hdf5_name}')
print(f'Ruta: {hdf5_path}')
print(f'Existe: {hdf5_path.exists()}')

if not hdf5_path.exists():
    raise FileNotFoundError(f'Archivo no encontrado: {hdf5_path}')

In [ ]:
try:
    measurement_list_dic = list_measurements(hdf5_path)
except FileNotFoundError:
    raise FileNotFoundError(f'Archivo no encontrado: {hdf5_path}')
except OSError as e:
    raise OSError(f'Archivo encontrado, pero no se puede abrir (posiblemente corrupto).\nError: {e}')

output_measurements = [m['name'] for m in measurement_list_dic if m['mode'] == 'output']

print(f'Output measurements ({len(output_measurements)}):')
for m in output_measurements:
    print(f'  {m}')

if len(output_measurements) == 0:
    raise ValueError('No hay output curves en este archivo HDF5.')

In [ ]:
def read_output_curves_with_direction(hdf5_path, measurement_name):
    curves = read_all_curves(hdf5_path, measurement_name)
    data_list = []
    
    for i, curve in enumerate(curves):
        v_gs = curve['v_gs']
        v_ds = curve['v_ds']
        i_ds = curve['i_ds']
        i_gs = curve.get('i_gs', np.zeros_like(i_ds))
        
        # Detectar dirección del sweep
        if len(v_ds) > 1:
            dv = np.diff(v_ds)
            if len(dv) > 0:
                sweep_dir = 'forward' if np.mean(dv > 0) > 0.5 else 'reverse'
            else:
                sweep_dir = 'forward'
        else:
            sweep_dir = 'forward'
        
        for j in range(len(v_ds)):
            data_list.append({
                'measurement': measurement_name,
                'curve_index': i,
                'V_GS (V)': v_gs[j] if j < len(v_gs) else v_gs[0],
                'V_DS (V)': v_ds[j],
                'I_DS (A)': i_ds[j],
                'I_GS (A)': i_gs[j] if j < len(i_gs) else 0,
                'sweep_direction': sweep_dir,
            })
    
    return pd.DataFrame(data_list)

print('Leyendo curvas output...')
dfs = []
for m_name in output_measurements:
    print(f'  {m_name}')
    df = read_output_curves_with_direction(hdf5_path, m_name)
    dfs.append(df)

df_output = pd.concat(dfs, ignore_index=True)
print(f'\nTotal de puntos: {len(df_output)}')
print(f'Columnas: {list(df_output.columns)}')
display(df_output.head(10))

## Visualización de curvas output

Gráficas de I_DS vs V_DS para diferentes V_GS.

In [ ]:
def plot_output_grid(hdf5_path, measurements, ncols=3, figsize_per=(5, 4)):
    n = len(measurements)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per[0]*ncols, figsize_per[1]*nrows))
    axes_flat = np.array(axes).flatten() if n > 1 else np.array([axes])
    
    for ax, m_name in zip(axes_flat, measurements):
        plot_output(hdf5_path, m_name, ax=ax)
    
    for ax in axes_flat[len(measurements):]:
        ax.set_visible(False)
    plt.tight_layout()
    return fig

fig = plot_output_grid(hdf5_path, output_measurements)
plt.show()

## Guardar CSV en data/processed/

In [ ]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

hdf5_stem = Path(hdf5_name).stem
csv_filename = f'output_{hdf5_stem}.csv'
csv_path = processed_dir / csv_filename

df_output.to_csv(csv_path, index=False)
print(f'CSV guardado en: {csv_path}')
print(f'Filas: {len(df_output)}')
print(f'Columnas: {list(df_output.columns)}')

## Registrar en SQLite

In [ ]:
def find_or_create_memory(hdf5_name, hdf5_rel_path, analysis_type):
    title = f'{analysis_type}: {hdf5_name}'
    hdf5_stem = Path(hdf5_name).stem
    
    existing = db.list_memories(search=hdf5_stem)
    for mem in existing:
        if mem['title'] == title:
            return mem['id'], False
    
    date_match = hdf5_stem[:10]
    try:
        datetime.strptime(date_match, '%Y-%m-%d')
        analysis_date = date_match
    except:
        analysis_date = datetime.now().strftime('%Y-%m-%d')
    
    substrate = ''
    if 'kapton' in hdf5_name.lower():
        substrate = 'Kapton'
    elif 'si' in hdf5_name.lower():
        substrate = 'Si/SiO2'
    
    memory_id = db.create_memory(
        title=title,
        type='experiment',
        subtype='session',
        date=analysis_date,
        description=f'{analysis_type} analysis from {hdf5_name}',
        substrate=substrate,
        process='electrical measurement',
        custom_fields={
            'hdf5_file': hdf5_name,
            'analysis_date': datetime.now().strftime('%Y-%m-%d'),
            'n_measurements': str(len(output_measurements)),
            'n_data_points': str(len(df_output)),
        }
    )
    
    db.add_reference(memory_id, 'hdf5', hdf5_rel_path)
    
    return memory_id, True

hdf5_rel_path = f'data/raw/{hdf5_name}'
memory_id, was_created = find_or_create_memory(hdf5_name, hdf5_rel_path, 'Output')

print(f'Memory ID: {memory_id}')
print(f'Created: {was_created}')
print(f'Title: Output: {hdf5_name}')

In [ ]:
csv_rel_path = f'data/processed/{csv_filename}'
db.add_reference(memory_id, 'csv', csv_rel_path)
print(f'CSV reference added: {csv_rel_path}')

analysis_id = db.save_analysis_result(
    memory_id=memory_id,
    function_name='extract_output_curves',
    params={
        'n_measurements': len(output_measurements),
        'n_data_points': len(df_output),
    },
    results_csv_path=csv_rel_path,
)

print(f'Analysis result saved with ID: {analysis_id}')

## Resumen

✅ **Procesamiento completado:**
- HDF5: `{hdf5_name}`
- Mediciones: {len(output_measurements)} output curves
- Puntos de datos: {len(df_output)}
- CSV guardado: `{csv_rel_path}`
- Memoria registrada en SQLite: ID {memory_id}

## Siguientes pasos en el dashboard

1. Abrir el dashboard: `python dashboard/app.py`
2. Buscar la memoria "Output: {hdf5_name}"
3. Visualizar curvas output (I_DS vs V_DS)
4. Comparar con memorias de transfer si están disponibles
5. Exportar resultados si es necesario